# Scene Occlusion

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
movedVarPcd = copy.deepcopy(varPcds[0]).translate(-varPosses[0])
newScene = drm.visualise_open3d_pointclouds(movedVarPcd)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
newScene.add_geometry(drm.o3d_mesh_to_trimesh(movedVarPcd.get_oriented_bounding_box()))
newScene.show()

In [ ]:
def build_occlusion_grid(
    reference: o3d.geometry.PointCloud,
    voxel_size: float = 0.05
) -> tuple:
    # --- Step 1: Translate point cloud so scanner is at origin ---
    pts_world    = np.asarray(reference.points)
    pts_centered = pts_world

    # --- Step 2: OBB local frame of the shifted cloud ---
    shifted_pcd = o3d.geometry.PointCloud()
    shifted_pcd.points = o3d.utility.Vector3dVector(pts_centered)
    obb    = shifted_pcd.get_oriented_bounding_box()
    R      = np.asarray(obb.R)
    center = np.asarray(obb.center)

    pts_local = (pts_centered - center) @ R

    # --- Step 3: Voxel indices ---
    min_bound_local = pts_local.min(axis=0)
    max_bound_local = pts_local.max(axis=0)
    grid_size       = np.floor((max_bound_local - min_bound_local) / voxel_size).astype(int) + 1

    voxel_indices   = np.floor((pts_local - min_bound_local) / voxel_size).astype(int)
    occupied_voxels = set(map(tuple, voxel_indices))

    # Scanner is at origin in shifted space — transform (0,0,0) into local frame
    origin_local = (np.zeros(3) - center) @ R
    origin_voxel = (origin_local - min_bound_local) / voxel_size

    # --- Step 4: Ray march from origin through each occupied voxel ---
    occluded_voxels = set()

    for voxel in occupied_voxels:
        voxel_f    = np.array(voxel, dtype=float) + 0.5
        ray_dir    = voxel_f - origin_voxel
        ray_length = np.linalg.norm(ray_dir)
        if ray_length == 0:
            continue
        ray_dir_n = ray_dir / ray_length

        t     = ray_length + 1.0
        t_max = ray_length + np.linalg.norm(grid_size)

        while t < t_max:
            current = np.floor(origin_voxel + t * ray_dir_n).astype(int)
            if np.any(current < 0) or np.any(current >= grid_size):
                break
            current_tuple = tuple(current)
            if current_tuple not in occupied_voxels:
                occluded_voxels.add(current_tuple)
            t += 1.0

    return occluded_voxels, occupied_voxels, obb, R, min_bound_local


def get_invisible_points_grid(
    points: o3d.geometry.PointCloud,
    reference: o3d.geometry.PointCloud,
    voxel_size: float = 0.05
) -> o3d.geometry.PointCloud:
    occluded_voxels, occupied_voxels, obb, R, min_bound_local = build_occlusion_grid(
        reference, voxel_size
    )

    center = np.asarray(obb.center)
    pts    = np.asarray(points.points)

    # Apply the same shift before transforming into local frame
    pts_local  = (pts - center) @ R
    voxel_idxs = [tuple(np.floor((p - min_bound_local) / voxel_size).astype(int)) for p in pts_local]

    invisible_mask = np.array([
        idx in occluded_voxels or idx in occupied_voxels
        for idx in voxel_idxs
    ])

    return points.select_by_index(np.where(invisible_mask)[0])


def visualise_occlusion_grid(
    occluded_voxels: set,
    occupied_voxels: set,
    obb: o3d.geometry.OrientedBoundingBox,
    R: np.ndarray,
    min_bound_local: np.ndarray,
    voxel_size: float = 0.05,
    show_occupied: bool = True,
    show_occluded: bool = True,
) -> trimesh.Scene:
    center = np.asarray(obb.center)

    unit_cube  = trimesh.creation.box(extents=[1, 1, 1])
    unit_verts = np.asarray(unit_cube.vertices)
    faces      = np.asarray(unit_cube.faces)

    def voxels_to_mesh(voxel_set: set, color: list) -> trimesh.Trimesh:
        if not voxel_set:
            return None
        indices       = np.array(list(voxel_set))
        centres_local = (indices + 0.5) * voxel_size + min_bound_local
        centres_world = centres_local @ R.T + center

        K         = len(indices)
        all_verts = (unit_verts[np.newaxis] * voxel_size + centres_world[:, np.newaxis, :]).reshape(-1, 3)
        offsets   = np.arange(K)[:, np.newaxis, np.newaxis] * 8
        all_faces = (faces[np.newaxis] + offsets).reshape(-1, 3)

        mesh = trimesh.Trimesh(vertices=all_verts, faces=all_faces, process=False)
        mesh.visual.face_colors = np.tile(
            np.array([int(c) for c in color], dtype=np.uint8),
            (len(all_faces), 1)
        )
        return mesh

    scene = trimesh.Scene()
    if show_occupied:
        m = voxels_to_mesh(occupied_voxels, color=[220, 60, 60, 180])
        if m is not None:
            scene.add_geometry(m, node_name="occupied")
    if show_occluded:
        m = voxels_to_mesh(occluded_voxels, color=[60, 120, 220, 120])
        if m is not None:
            scene.add_geometry(m, node_name="occluded")

    return scene

In [ ]:
occluded_voxels, occupied_voxels, obb, R, min_bound_local = build_occlusion_grid(
    movedVarPcd, voxel_size=THRESHOLD_RESOLUTION
)

scene = visualise_occlusion_grid(
    occluded_voxels, occupied_voxels,
    obb, R, min_bound_local,
    voxel_size=THRESHOLD_RESOLUTION,
)


In [ ]:

# Optionally overlay the point cloud
cloud = drm.o3d_pointcloud_to_trimesh(varPcds[0])
scene.add_geometry(cloud, node_name="cloud")

scene.show()

## Experiments